
Geo Data Science with Python,
Prof. Susanna Werth, VT Geosciences

# Time Frequency Analysis 


This notebook is accompanied by the lecture L09 presentation slides.


---


Content:
-------
- **A.** Non-stationary signals and FFT 
- **B.** Wavelet time frequency analysis
- **C.** Decomposing the El Nino Index
--- 

In [ ]:

# On Google Colab, install the following packages...

# ! pip install PyWavelets


In [ ]:

# Import standard packages
import requests
import numpy as np
import matplotlib.pyplot as plt
import scipy
import pandas as pd

# let's set a standard size for figures (a bit smaller, so they fit on the screen)
from matplotlib import rcParams
rcParams['figure.figsize'] = [4, 3]
rcParams.update({'font.size': 10})


---
# A. Non-stationary signals and FFT

A signal is non-stationary if its statistical properties (mean, variance, amplitude, frequency, etc.) change over time. Let's create some synthetic stationary and non-stationary time series.


### Generate Synthetic Time Series Data


In [ ]:

# Create a signal: combination of annual and semi-annual sine waves + noise
fs2 = 365  # sampling frequency (1/year = 31.5 Mega Hz)
T2 = 6.0     # duration (years)
t2 = np.linspace(0, T2, int(fs2*T2))

# signal frequencies, amplitude, and noise components 
f1, f2, f6 = 1, 1/3, 8     # frequencies of sine waves
f1_nonSt = np.linspace(1, 5, len(t2))

signal2 = 1   * np.sin(2*np.pi * f1 * t2) + 0.6 * np.sin( 2*np.pi * f2 * t2 ) # Stationary: 1 & 3 year wave
signal3 = 1   * np.sin(2*np.pi * f1 * t2) + 1   * np.sin( 2*np.pi * f6 * t2 ) # Stationary: 0.1 & 1 year waves
signal4 = 1   * np.sin(2*np.pi * f1 * t2)         # Annual wave
signal5 = 1/2 * np.sin(2*np.pi * f1 * t2) * t2    # Increasing amplitude
signal6 = 1   * np.sin(2*np.pi * f1_nonSt * t2)   # Changing frequency (from 1/year to 5/year)
noise   = 0.8 * np.random.randn(len(t2))          # Noise

signal2_noise = signal2 + noise
signal3_noise = signal3 + noise
signal4_noise = signal4 + noise
signal5_noise = signal5 + noise
signal6_noise = signal6 + noise


**Task**: Which of the signals above are non-stationary?

### Plot synthetic data

In [ ]:

# Plot sythetic signals without noise
plt.figure(figsize=(8,3))
plt.plot(t2, signal2+8,  label='Stationary: 1 & 3 year wave')
plt.plot(t2, signal3+6,label='Stationary: 0.1 & 1 year waves')
plt.plot(t2, signal4+4,label='Annual wave')
plt.plot(t2, signal5+2,label='Nonstationary amplitude')
plt.plot(t2, signal6,label='Nonstationary frequency')
plt.title("Sythetic Signals")
plt.xlabel("Time [year]")
plt.ylabel("Amplitude")
plt.xlim([-0.2,8.5])
plt.legend(loc='upper right',fontsize=6,frameon=False)
plt.show()


In [ ]:

# Plot sythetic signals with noise
plt.figure(figsize=(6,2))
plt.plot(t2, signal4_noise,label='Annual wave')
plt.plot(t2, signal5_noise+2,label='Nonstationary amplitude')
plt.plot(t2, signal6_noise+4,label='Nonstationary frequency')
plt.title("Sythetic Signals")
plt.xlabel("Time [year]")
plt.ylabel("Amplitude")
plt.legend(loc='lower left',fontsize=6,frameon=False)
plt.show()


## Discrete FFT

In [ ]:

# Compute Fourier Transform using scipy or numpy fft package:
fhatA = scipy.fft.fft(signal4_noise)     
fft_freqA = scipy.fft.fftfreq(len(t2), 1/fs2) 
freqsA = fft_freqA[fft_freqA > 0]
amplitudeA = np.abs(fhatA[fft_freqA > 0]) * 2 / len(t2) 

fhatB = scipy.fft.fft(signal5_noise)     
fft_freqB = scipy.fft.fftfreq(len(t2), 1/fs2) 
freqsB = fft_freqB[fft_freqB > 0]
amplitudeB = np.abs(fhatB[fft_freqB > 0]) * 2 / len(t2) 

fhatC = scipy.fft.fft(signal6_noise)     
fft_freqC = scipy.fft.fftfreq(len(t2), 1/fs2) 
freqsC = fft_freqC[fft_freqC > 0]
amplitudeC = np.abs(fhatC[fft_freqC > 0]) * 2 / len(t2) 

# Plot the signal spectrum
fig, ax = plt.subplots(1,3,figsize=(10,2))
ax[0].stem(freqsA, amplitudeA, basefmt=" ")
ax[1].stem(freqsB, amplitudeB, basefmt=" ",linefmt='orange')
ax[2].stem(freqsC, amplitudeC, basefmt=" ",linefmt='green')
[a.set_xscale('log') for a in ax[0:3]]
ax[1].set_title("Amplitude spectrum for nonstationary signals over f")
ax[1].set_xlabel("Frequency [31MHz = 1/year]")
ax[0].set_ylabel("Amplitude")
plt.show()


**Task**: For signal 6 use the noise free time series. Does that help with resolving the spectra?


## FFT over a moving window: Periodogram

In [ ]:

# FFT- windowed, but no averaging
fA, PsdA = scipy.signal.periodogram(signal2_noise, fs2, window='hann') # options: boxcar, hann
fB, PsdB = scipy.signal.periodogram(signal3_noise, fs2, window='hann')
fC, PsdC = scipy.signal.periodogram(signal4_noise, fs2, window='hann')
deltaf = fs2/len(t2)  # frequency resolution 
# PSD = amplitude^2/frequency/2     (factor 2 accounts for using a one-sided spectrum, for real signals).
AmpA = np.sqrt(2 * PsdA * deltaf)
AmpB = np.sqrt(2 * PsdB * deltaf)
AmpC = np.sqrt(2 * PsdC * deltaf)

plt.figure(figsize=(4,2))
plt.plot(fA, AmpA,label='A')
plt.plot(fB, AmpB+0.1,label='B')
plt.plot(fC, AmpC-0.1,label='C')
plt.xscale('log')
plt.xlabel('Frequency (1/year)')
plt.ylabel('Amplitude (m)')
plt.legend(loc=0,frameon=False)
plt.show()


## Windowed FFT with Overlap: Spectrogram via STFT

The Short-Time Fourier Transform (STFT) computes a moving-windowed FFT, which allows a time-frequency representation in a spectogram.

Windowing doesn’t make the Fourier Transform continuous in frequency, but it makes it locally continuous in time by applying the FFT on short, overlapping time segments. This givesa time-resolved (or quasi-continuous) view of frequency content.


In [ ]:
# Signal sampling frequency: 365/year;  sample length: 6*365 = 2190

y = signal3  # choose time series to analyze

f, t, Zxx = scipy.signal.stft(y, fs=fs2, window='hann', nperseg=1200, noverlap=1100)  
# nperseg/noverlab: for lower resolution use 400/200, for higher resolution use 1200/1100
#   nperseg = number of data points in each segment, have at least 4–6 segments for reliable averaging
#      small: more segments, smoother, but poorer frequency resolution (can miss long-period features).
#      large: high frequency resolution, fewer segments, noisier spectrum.
#   noverlap: use 50–75% overlap with a Hann window

amp = 2*np.abs(Zxx)   # or use 4*np.abs(Zxx)**2 for power

plt.figure(figsize=(6,3))
extent = [t.min(), t.max(), f.min(), f.max()] # generate axes label for imshow
plt.imshow(amp, aspect='auto', origin='lower', extent=extent)
plt.colorbar(label='Amplitude (m)')
plt.title('Windowed FFT: Spectrogram')
plt.xlabel('Time (year)')
plt.ylabel('Frequency (1/year)')
plt.ylim([0,20]) # limit this for better visibility of low frequencies present in our signal
plt.show()


--- 
# Exercise A

1. Discuss the output of the Spectogram for signals n=2...6 (`signaln`). For signal 2, change the resolution to resolve both frequencies present in the signal (see comments in the code on parameter `nperseg` and `noverlap`)

2. How does the Spectogram change, if noise is added to the signals? (`signaln_noise`)

---
# B. Continuous Time-Frequency Analysis with Wavelets

Wavelet analysis is a method that decomposes a time series into time–frequency space, allowing you to identify how the dominant frequencies or periodicities of a signal vary over time. Unlike Fourier analysis, which gives global frequency content, wavelet analysis preserves both temporal and spectral information, making it ideal for studying non-stationary signals.

Here we are using the package pywt:
https://pywavelets.readthedocs.io/en/


PyWavelets (pywt) is intentionally designed to abstract away much of the mathematical detail of the wavelet transform. It automates the scaling, shifting, normalization, and frequency mapping so that users can get correct transforms quickly, although, somewhat at the cost of transparency and flexibility.

In [ ]:
import pywt

## Wavelet Functions

In [ ]:
# See available wavelets
print(pywt.wavelist(kind='continuous')) # all continuous wavelets

In [ ]:
# Continuous Wavelets

plt.figure(figsize=(6,3))
for name in ['cmor1.5-1.0', 'morl', 'cgau3','mexh']:  # choose wavelet: morl, gaus8, shan, mexh, cmor1.5-1.0, cgau3
    w = pywt.ContinuousWavelet(name)
    psi, x = w.wavefun(length=512)
    plt.plot(x, np.real(psi), label=name) 
    # plt.plot(x, np.imag(psi), label='Imag part') # plot imaginary part
plt.legend(); plt.title('Different Continuous Wavelets')
plt.show()


## Continuous Wavelet Transform

In [ ]:

y = signal3  # choose time series to analyze

# pick scales to cover your frequency band of interest
scales = np.arange(0.1, 365*3)  # in periods, extend for lower frequencies, at 1.5 calculation is faster though
dt = 1/fs2           # sampling frequency
wave = 'cmor1.5-1.0' # wavelet function, other options are: cmor1.5-1.0, morl, gaus8, mexh

# Estimate continuous wavelet transform
coefficients, frequencies = pywt.cwt(y, scales=scales, wavelet=wave, sampling_period=dt) # complex Morlet (sharper than real Morlet)

# Wavelet power spectrum
power = (np.abs(coefficients))**2  / scales[:, None]  # Raw |coef|^2 favors larger scales. Normalize to compare peaks at low scales fairly.         
#power = (np.abs(coefficients))**2       

# Convert frequency (1/year) to period (years) for intuitive y-axis
periods = scales * dt   # also: periods = 1.0 / frequencies


#### Wavelet Power Spectrum

In [ ]:

# Plotting of the power spectra
plt.figure(figsize=(6, 3))
extent = [t2[0], t2[-1], periods.min(), periods.max()]  # generate axes label for imshow
plt.imshow(power, aspect='auto', cmap='plasma', extent=extent, origin='lower')
plt.colorbar(label="Power")
plt.ylabel("Scale: Period (years)")
plt.xlabel("Time (years)")
plt.title("Wavelet Power Spectrum (PyWavelets CWT)")
plt.show()


#### Amplitude and Phase of the Coefficients

In [ ]:

# Plotting of the coefficients amplitude and phase

fig, ax = plt.subplots(1,2, figsize=(8, 3), constrained_layout=True)
extent = [t2[0], t2[-1], periods.min(), periods.max()]  # generate axes label for imshow

im0 = ax[0].imshow(np.real(coefficients), aspect='auto', cmap='plasma', extent=extent, origin='lower')
cbar0 = fig.colorbar(im0, ax=ax[0])
ax[0].set_ylabel("Scale: Period (years)")
ax[0].set_xlabel("Time (year)")
ax[0].set_title("Amplitude")

im1 = ax[1].imshow(np.angle(coefficients) , aspect='auto', cmap='plasma', extent=extent, origin='lower')
cbar1 = fig.colorbar(im1, ax=ax[1])
cbar1.set_label("(radians: –π to +π ≈ –3.14 to +3.14)")
ax[1].set_title("Phase ")
ax[1].set_xlabel("Time (year)")
plt.show()

**Task**: Estimate the wavelet power spectra, amplitude and phase for signal3 using a Morlet real (e.g., morl) versus complex (e.g., cmor1.5-1.0) wavelet base functions. What differences do you observe? (This is equal to Exercise B.1)

## Cone of Influence

The cone of influence (COI) marks regions near the edges of a time series where wavelet results become unreliable due to truncation or zero-padding at the edges. Its extent depends on both the wavelet type, which determines the effective temporal width (e.g., Morlet ≈ √2 × scale), and the length of the time series, with shorter records or larger scales producing a wider COI at longer periods.

In [ ]:
# Adding a manually calculated cone of influence to the graph

# Plotting
plt.figure(figsize=(6, 3))
extent = [t2[0], t2[-1], periods.min(), periods.max()]  # generate axes label for imshow
plt.imshow(power, aspect='auto', cmap='plasma', origin='lower', extent=extent)
plt.colorbar(label="Power")
plt.ylabel("Scale: Period (years)")
plt.xlabel("Time (years)")
plt.title("WPS & COI")

# Estimate cone of Influence (COI)
#    half-width for each scale: k * scale * dt, with k_coi = np.sqrt(2.0)  for morlet
dt_years = float(np.mean(np.diff(t2)))  # sampling period in years
coi_half = np.sqrt(2.0) * scales * dt_years  

# Time coordinates of the left and right COI boundaries for each scale (in years).
t_left  = t2[0]  + coi_half    # left COI boundary vs scale
t_right = t2[-1] - coi_half    # right COI boundary vs scale

ax = plt.gca()

# Shade outside COI
ax.fill_betweenx(periods, t2[0], t_left,  color='k', alpha=0.35, linewidth=0)
ax.fill_betweenx(periods, t_right, t2[-1], color='k', alpha=0.35, linewidth=0)

# Draw dashed COI boundary lines
ax.plot(t_left,  periods,  'w--', lw=1.2)
ax.plot(t_right, periods, 'w--', lw=1.2)

ymaxl = min(2.15, scales.max()/365)
plt.ylim( [0, ymaxl])
plt.show()

**Note**: See the supplement below for example or information on more relevant approaches (inverse WT, coherence analysis, Welch FT)

---
# Exercise B


1. Estimate the wavelet power spectra, amplitude and phase for signal3 using a Morlet real (e.g., morl) versus complex (e.g., cmor1.5-1.0) wavelet base functions. What differences do you observe?

2. Estimate and discuss the wavelet power spectra (WPS) for signals n=2...6 (`signaln`).

3. Compute the wavelet power spectrum (WPS) of a step function using two different real continuous wavelets: the Morlet (`morl`) and the Mexican Hat (`mexh`). Compare the resulting time–frequency representations and discuss whether the analysis depends on the choice of wavelet base function. Which wavelet resolves the step feature more clearly? What is the implication for signals of more harmonic or more transient variations?

```python
    signal_step = np.ones(len(t2))
    signal_step[~(t2>T2/2)] -= 2
```

**Extra Credit**

4. How does the WPS change, if noise is added to the signals? (`signaln_noise`)


---
# C. Decomposing the El Nino Index

## Data: El Nino Southern Oscillation (ENSO) Index: Nino 3.4

In [ ]:

# Load time series of Nino Anom 3.4 Index 

# Option A
url = ('https://psl.noaa.gov/data/correlation/nina34.anom.csv')
# Nino Anom 3.4 Index using ersstv5 from CPC, missing value -99.99 
# Source Info: https://psl.noaa.gov/data/timeseries/month/Nino34_CPC/

# Option B
# url = ('https://psl.noaa.gov/data/timeseries/month/data/nino34.long.anom.csv')
# NINA34, missing value -99.99 
# Source Info: https://psl.noaa.gov/data/timeseries/month/DS/Nino34/

dfNino34 = pd.read_csv(url)
dfNino34 = dfNino34.replace(-9999.0, np.nan)
dfNino34.columns.values[1] = 'Nino34'
dfNino34 = dfNino34.dropna()

dfNino34['Date'] = pd.to_datetime(dfNino34['Date'], format='%Y-%m-%d')

dfNino34.head() # Show the first few rows of the dataframe to check its structure and content


In [ ]:

# Sampling parameter
N = len(dfNino34['Nino34']) #length of time series
fsN = 12 # sampling frequency (1/year)
print("Time series length =", N,"; Sampling Frequency:", fsN,"cycles/year")

In [ ]:

# Plot the Nino 3.4 time series
plt.figure(figsize=(10,3))
plt.plot(dfNino34['Date'], dfNino34['Nino34'])
plt.title("Nino 3.4 Index")
plt.xlabel("Time")
plt.ylabel("Index Value")
plt.show()

---
# Exercise C

1. Using the monthly Niño 3.4 index time series:

- a) Perform a Fast Fourier Transform (FFT) on the El Niño 3.4 Index time series. Plot the resulting power spectrum (frequency or period vs. spectral power).
- b) Identify the dominant frequencies or periods where most of the signal power lies.
- c) Evaluate how reliable these frequency estimates are for describing the signal’s variability. Discuss whether the time series appears stationary or non-stationary, and explain why the FFT may or may not be appropriate for such data.

2. Using the monthly Niño 3.4 index time series:
- a) Compute a continuous wavelet transform (CWT) with a complex Morlet-type (cmor1.5-1.0) wavelet and plot the wavelet power spectrum (WPS) twice: (a) without the cone of influence (COI) and (b) with the COI overlaid. Since we are looking out for lower frequency, I suggest not to scale power by scales. Try to label the x-axis with calendar years.
- b) Discuss the WPS: In which period bands (≈ frequencies) does most power occur, and how does it change over time? Do you observe on/off bursts or transitions in the dominant period? Interpret these features in terms of ENSO variability.

3. Deepen your knowledge about the meaning of the Index. Investigate or ask an LMM for an explanation of the El Nino Southern Oscillation (ENSO) phenomenon. Then:

- a) Compile a 2-3 sentence definition of the El Nino 3.4 index (note the index not the ENSO phenomenon), including the value range and the difference between El Nino and La Nina.
- b) Do El Nino and La Nina always occur in succession? What does this mean for the stationarity of the Nino Index?
- c) The wavelet power spectrum (WPS) of the complex cmorlet wavelet only shows the strength of variability over time and frequency but cannot distinguish between warm (El Niño) and cold (La Niña) phases. Explain what you need to plot in order to identify the timing and sign (warming vs. cooling) of these phases, show an example, and describe how this information differs from what is shown in the WPS.

Extra Credit:

4. Find out if and how the ENSO influences the weather pattern in a) Virginia and b) your home region. (If your home region is Virginia, choose any other place around the world you are interested in.) Consider both cool (El Nino) and warm (La Nina) events. Report back a brief summary.



---
# Supplement: Additional Approaches

## Inverse CWT

Similar to FFT, power spectrum of the wavelet transform can reconstructed, and combined with spectral filtering. This allows decomposition of non-stationary signals into specific frequencies, noise reduction or coherence analysis.

In the current version of PyWavelets (pywt), there is no built-in icwt or inverse continuous wavelet transform function.

However, other packages are more comprehensive and are suggested if you want to continue working with ICWT. Especially the software from Torrence & Compo (1998) includes tutorials comprehensive analysis plots is very useful and recommended. This was originially developed in MATLAB, while the Python version is still being optimized:
- **MATLAB**: https://paos.colorado.edu/research/wavelets/software.html
- **PyCWT** https://pycwt.readthedocs.io

> Torrence, C., Compo, G.P. (1998) A practical guide to wavelet analysis, Bull. Am. Meteorol. Soc., 79, 61–78, https://doi.org/10.1175/1520-0477(1998)079<0061:APGTWA>2.0.CO;2

Another option is the package **PyTorch**. This blog provides an application example: https://blogs.rstudio.com/ai/posts/2023-01-19-torchwavelets.

These packages allows also more insight into exact mathematical normalization and more parameters for fine control of the transform and statistically correct estimation of COI and statistical significance assessments, which is relevant for exact interpretable geophysical or hydrologic analysis (where normalization and power scaling matter).

## Coherence Analysis

In [ ]:

x = signal4  # annual wave
y = signal3  

# Coherence Analysis (based on Windowed FFT using Welch method, similar to Periodogram, see supplement below)
f, Cxy = scipy.signal.coherence(x, y, fs=fs2, nperseg=300)

plt.figure(figsize=(6,3))
plt.semilogx(f, Cxy)
plt.xlabel('Frequency [cycles/year]')
plt.ylabel('Coherence')
plt.title('Magnitude-Squared Coherence')
plt.grid(True)
plt.show()


Note, the package **PyCWT** following the Torrence & Compo (1998) formulation also includes assessment of coherence for nonstationary signals using wavelets.

Another option is the wavelet coherence package in **MATLAB** from Grinsted et al., 2004: https://noc.ac.uk/business/marine-data-products/cross-wavelet-wavelet-coherence-toolbox-matlab

> Grinsted, A., Moore, J.C., Jevrejeva, S. (2004) Application of the cross wavelet transform and wavelet coherence to geophysical time series, Nonlin. Processes Geophys., 11, 561–566, https://doi.org/10.5194/npg-11-561-2004

## Welch FT

Estimates the windowed FT power spectrum of a signal by averaging modified periodograms of overlapping, windowed segments to reduce noise and variance.

In [ ]:

nperseg = 365 * 5
# nperseg = number of data points in each segment.
# small: larger and fewer f bins (poorer frequency resolution), but more averaging & smoother PSD
# large: narrower and more f bins (higher frequency resolution), noisier PSD estimate    
f3, Pxx3 = scipy.signal.welch(signal4_noise, fs=fs2, nperseg=nperseg)
f4, Pxx4 = scipy.signal.welch(signal5_noise, fs=fs2, nperseg=nperseg)
f5, Pxx5 = scipy.signal.welch(signal6_noise, fs=fs2, nperseg=nperseg)

plt.plot(f3, Pxx3,label='A')
plt.plot(f4, Pxx4,label='B')
plt.plot(f5, Pxx5,label='C')
plt.xscale('log')
#plt.yscale('log')
plt.xlabel('Frequency (1/year)')
plt.ylabel('Power (m$^2$/year$^2$)')
plt.legend(loc=0,frameon=False)